# Short-Term Reversal (1-Month) — Strategy Research Notebook

This notebook asks a specific research question:

> Does buying the past month's biggest losers and shorting the past month's biggest winners, within the S&P 500, earn a positive return net of transaction costs when rebalanced monthly?

In plain English: when a stock falls a lot in one month, part of that fall is often an overshoot — sellers who needed liquidity pushed the price below fair value. If that is true, recent losers should bounce back a little, and recent winners should give back a little. We test whether that bounce is large enough and reliable enough to trade after costs.

Example:

```text
It is the last trading day of March.

Stock A fell 18% during March  -> it ranks near the TOP of our signal (biggest loser).
Stock B rose 25% during March  -> it ranks near the BOTTOM of our signal (biggest winner).

The strategy for April:
  LONG  the top 20% of the ranking  (stocks like A, the biggest losers)
  SHORT the bottom 20% of the ranking (stocks like B, the biggest winners)

Positions are equal-weighted within each leg, held for one month, then re-ranked.
```

This strategy is registered in the platform as `short_term_reversal` in `core/strategies/registry.py`, and its factor column is `rev_21d` in the factor panel. All computation is imported from `core/` — no strategy math is re-implemented in this notebook.

**References**: Jegadeesh (1990), Lehmann (1990) for the original evidence; Nagel (2012) "Evaporating Liquidity" for why the premium weakened after electronic market-making.

## 1. Setup And Data Loading

Paths are resolved from the repository root so this notebook works whether Jupyter was started in the repo root or inside `notebooks/`. If the required data files are missing, we stop with a clear message instead of failing later with a confusing error.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve the repo root whether Jupyter started in the root or in notebooks/.
_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "core").is_dir() else _cwd.parent
assert (REPO_ROOT / "core").is_dir(), f"Could not locate repo root from {_cwd}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.backtest.portfolio import sp500_universe_filter
from core.data.factors.build_factors import short_term_reversal
from core.metrics.performance import (
    calculate_cumulative_returns,
    calculate_performance_metrics,
)
from core.strategies import get_strategy, run_factor_cross_section_backtest

FACTORS_PATH = REPO_ROOT / "data" / "factors" / "factors_price.parquet"
PRICES_PATH = REPO_ROOT / "data" / "factors" / "prices.parquet"

missing = [p for p in (FACTORS_PATH, PRICES_PATH) if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing data files: {[str(p) for p in missing]}. "
        "Run scripts/backfill_all.py (or restore data/factors/) before executing this notebook."
    )

factor_panel = pd.read_parquet(FACTORS_PATH)
adjusted_close_prices = pd.read_parquet(PRICES_PATH)
PANEL_TZ = adjusted_close_prices.index.tz

print("factor panel:", factor_panel.shape, "columns:", list(factor_panel.columns))
print("price panel: ", adjusted_close_prices.shape, "tz:", PANEL_TZ)

## 2. What Exactly Are We Observing?

The primary observed series is the **daily net return of a long/short equity portfolio**: arithmetic daily returns, equal-weighted within each leg, long leg minus short leg, minus transaction costs. It is produced entirely by `run_factor_cross_section_backtest` in `core/strategies/factor_runner.py`.

Input data sources:

- **Adjusted-close stock prices** (`data/factors/prices.parquet`): wide panel, one column per symbol, daily frequency, timezone-aware index (America/New_York). Prices are yfinance-adjusted for splits and dividends.
- **Factor panel** (`data/factors/factors_price.parquet`): MultiIndex `(date, symbol)`; we use only the `rev_21d` column.
- **Point-in-time S&P 500 membership** (`sp500_universe_filter()`): on each date, only symbols that were actually in the index on that date are eligible. This removes survivorship bias — we do not let today's index membership leak into the past.

What the model does **not** see:

- No fundamentals, no news, no volume, no intraday data.
- No borrow costs or short-availability constraints (the short leg is simulated as free to borrow).
- No future information: the signal used on day *t* comes from prices up to day *t-1* (a 1-trading-day lag is applied by the backtest engine, `signal_lag_days=1`).

### Per-Variable Audit

#### `rev_21d`

- **Source**: computed by `short_term_reversal()` in `core/data/factors/build_factors.py` from adjusted-close prices; stored in `data/factors/factors_price.parquet`.
- **Transformation**: the negative trailing 21-trading-day return, `-(P(t) / P(t-21) - 1)`. The sign is flipped so that a stock that FELL over the past month gets a HIGH value. Ranking descending and buying the top tier therefore buys losers. Units: decimal return (0.10 means the stock lost 10% over the window).
- **Missing-data handling**: if either `P(t)` or `P(t-21)` is missing, the value is NaN — no fill of any kind. NaN rows are excluded from the ranking on that date.
- **Publication lag**: none in construction (uses closes observed end-of-day); the backtest engine additionally lags the signal by 1 trading day before it becomes actionable.
- **Standardization timing**: none. The factor is used as a raw cross-sectional ranking each day; no scaler is fit.
- **Leakage risk**: low. The window is strictly trailing, and the 1-day execution lag means the close used to build the signal is earlier than the close used to enter the position.

#### `adjusted_close_prices` (return engine input)

- **Source**: `data/factors/prices.parquet`, built by `scripts/backfill_all.py` / `scripts/update_daily.py` from yfinance.
- **Transformation**: daily arithmetic returns via `pct_change(fill_method=None)` inside `calculate_portfolio_returns`.
- **Missing-data handling**: NaN prices are never forward-filled. A held position whose price disappears is realised at -100% on the long leg (+100% on the short leg) — the bankruptcy convention.
- **Publication lag**: none; closes are known end-of-day.
- **Standardization timing**: not applicable.
- **Leakage risk**: low for the mechanism, but see the data-quality limitation in Section 6 — a small set of symbols has corrupted price spikes that can distort long-window results.

## 3. Signal Sanity Check: Does `rev_21d` Mean What We Think?

Before backtesting, verify the sign convention on a real stock: pick a symbol, compute `rev_21d` by hand from prices, and confirm it matches the stored panel. A month where the stock fell must produce a positive `rev_21d`.

In [ ]:
check_symbol = "AAPL"
# IMPORTANT: recompute on the raw panel column WITHOUT dropna. The stored factor
# shifts 21 rows over the full panel calendar; dropping NaN gaps first would
# compress the window and produce different values near missing dates.
aapl_close_panel = adjusted_close_prices[check_symbol]

rev_recomputed = short_term_reversal(aapl_close_panel, window=21)
rev_stored = factor_panel.xs(check_symbol, level="symbol")["rev_21d"].dropna()

# Align on shared dates and confirm the stored panel matches a fresh computation.
shared_dates = rev_recomputed.dropna().index.intersection(rev_stored.index)
max_abs_diff = (rev_recomputed.loc[shared_dates] - rev_stored.loc[shared_dates]).abs().max()
print(f"{check_symbol}: {len(shared_dates)} shared dates, max |recomputed - stored| = {max_abs_diff:.2e}")
assert max_abs_diff < 1e-10, "Stored rev_21d does not match a fresh computation from prices"

# Sign convention: the trailing 21-row return and the signal must be exact opposites.
trailing_21d_return = aapl_close_panel.pct_change(21, fill_method=None).loc[shared_dates]
worst_month_date = trailing_21d_return.idxmin()
print(
    f"Worst 21d stretch for {check_symbol}: {worst_month_date.date()} "
    f"(return {trailing_21d_return.min():+.1%}) -> rev_21d = {rev_stored.loc[worst_month_date]:+.4f} (should be positive)"
)
assert rev_stored.loc[worst_month_date] > 0

## 4. Portfolio Construction Disclosure

There are no regimes or model-inferred labels in this strategy — the only "decision" is the cross-sectional ranking. Here is the exact mechanical rule:

```text
On each monthly rebalance date t (last business day of the month):

1. Take rev_21d observed at close(t-1)         <- 1-trading-day lag, no same-close trading
2. Keep only symbols that were in the S&P 500 on date t (point-in-time membership)
3. Drop NaN / non-finite values and |rev_21d| >= 10 (data-error guard)
4. Require at least 20 valid symbols, otherwise no positions that month
5. Rank descending:
     top 20%    -> signal +1  (biggest losers  -> LONG)
     bottom 20% -> signal -1  (biggest winners -> SHORT)
6. Equal-weight within each leg; hold until the next rebalance
7. Charge 10 bps (0.10%) of turnover as transaction cost at each rebalance
```

When this notebook says "the strategy is long a stock", that means exactly: the stock's `rev_21d` value, observed one trading day before the rebalance, ranked in the top 20% among that day's point-in-time S&P 500 members with valid data. Nothing else — no fundamentals, no discretion.

Both legs are fully invested at every rebalance (100% long / 100% short of portfolio notional); there is no volatility targeting or exposure overlay in this notebook.

In [ ]:
# The registry entry is the single source of truth for what this strategy claims to be.
strategy_metadata = get_strategy("short_term_reversal")
print(strategy_metadata.title)
print()
print("Hypothesis:", strategy_metadata.hypothesis)
print()
print("Expected Sharpe range (published, gross-of-frictions era):", strategy_metadata.expected_sharpe_range)

## 5. Backtest

We run the shared platform pipeline (`run_factor_cross_section_backtest`) over 2018–2025. We deliberately use a recent window because the price panel contains corrupted quotes for a small set of delisted symbols concentrated in 2010–2017 (documented in Section 6); the recent window keeps the result interpretable.

We compare three things:

1. **Reversal long/short, net of 10 bps costs** — the actual strategy.
2. **Reversal long/short, gross (0 costs)** — same trades, no costs. The gap between 1 and 2 is the cost drag, which the literature says is the reversal strategy's main killer.
3. **12-1 momentum long/short, net** — the platform's reference factor on the identical pipeline, so any difference is the signal, not the plumbing.

In [ ]:
BACKTEST_START = pd.Timestamp("2018-01-01", tz=PANEL_TZ)
BACKTEST_END = pd.Timestamp("2025-12-31", tz=PANEL_TZ)
TRANSACTION_COST = 0.001  # 10 bps of turnover per rebalance

sp500_membership = sp500_universe_filter()

backtest_runs = {
    "reversal_net": dict(factor_col="rev_21d", transaction_cost=TRANSACTION_COST),
    "reversal_gross": dict(factor_col="rev_21d", transaction_cost=0.0),
    "momentum_net": dict(factor_col="mom_12_1", transaction_cost=TRANSACTION_COST),
}

daily_net_returns = {}
for run_name, overrides in backtest_runs.items():
    daily_net_returns[run_name] = run_factor_cross_section_backtest(
        factor_panel,
        adjusted_close_prices,
        start=BACKTEST_START,
        end=BACKTEST_END,
        top_pct=0.20,
        bottom_pct=0.20,
        long_only=False,
        rebalance_freq="ME",
        min_stocks=20,
        signal_lag_days=1,
        universe_filter=sp500_membership,
        **overrides,
    )
    print(f"{run_name}: {len(daily_net_returns[run_name])} trading days")

In [ ]:
metrics_by_run = pd.DataFrame(
    {name: calculate_performance_metrics(rets) for name, rets in daily_net_returns.items()}
).T[["annualized_return", "annualized_volatility", "sharpe_ratio", "sortino_ratio", "max_drawdown", "calmar_ratio"]]

metrics_by_run.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for run_name, rets in daily_net_returns.items():
    wealth_index = calculate_cumulative_returns(rets)
    ax.plot(wealth_index.index, wealth_index.values, label=run_name, linewidth=1.2)
ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--")
ax.set_title("Growth of $1 — reversal (net vs gross) and momentum reference, 2018–2025")
ax.set_ylabel("wealth index (start = 1.0)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Decision

We are comparing the **net short-term reversal long/short strategy** against two references: the same trades with zero costs (to isolate cost drag), and the platform's 12-1 momentum strategy on the identical pipeline (to see whether reversal adds anything momentum does not).

The conclusion is decided by these yes/no questions, all on the 2018–2025 window:

- **Is the net Sharpe ratio positive?** If no, the strategy loses money per unit of risk after costs and is not deployable as-is.
- **Is the gross Sharpe ratio positive?** If yes while net is negative, the signal contains information but costs destroy it — a faster rebalance or cost reduction could revive it. If gross is also negative, the signal itself is dead in this universe and period.
- **Is the net Sharpe higher than the momentum reference?** If no, there is no reason to prefer reversal over the factor the platform already ships.

Sign convention for the delta computed below: `delta = reversal_net - momentum_net`, so a **positive** delta means reversal is better on that metric (for `max_drawdown`, which is negative-valued, a positive delta also means "less bad").

Metrics kept for context only (they do not decide the conclusion): Sortino, Calmar, volatility. They describe the shape of the risk, not whether the premium exists.

In [ ]:
decision_metrics = ["sharpe_ratio", "annualized_return", "max_drawdown"]

reversal_vs_momentum_delta = (
    metrics_by_run.loc["reversal_net", decision_metrics]
    - metrics_by_run.loc["momentum_net", decision_metrics]
)
cost_drag = (
    metrics_by_run.loc["reversal_gross", ["sharpe_ratio", "annualized_return"]]
    - metrics_by_run.loc["reversal_net", ["sharpe_ratio", "annualized_return"]]
)

print("Q1  Net Sharpe positive?          ",
      "YES" if metrics_by_run.loc["reversal_net", "sharpe_ratio"] > 0 else "NO",
      f"({metrics_by_run.loc['reversal_net', 'sharpe_ratio']:.2f})")
print("Q2  Gross Sharpe positive?        ",
      "YES" if metrics_by_run.loc["reversal_gross", "sharpe_ratio"] > 0 else "NO",
      f"({metrics_by_run.loc['reversal_gross', 'sharpe_ratio']:.2f})")
print("Q3  Beats momentum reference?     ",
      "YES" if reversal_vs_momentum_delta["sharpe_ratio"] > 0 else "NO",
      f"(delta Sharpe {reversal_vs_momentum_delta['sharpe_ratio']:+.2f})")
print()
print("Cost drag (gross minus net):")
print(cost_drag.round(4).to_string())

## 7. Disclosed Risks And Limitations

- **The premium is likely gone in this universe.** Published reversal profits are concentrated pre-2000, in small caps, at weekly horizons. We test post-2018 S&P 500 large caps at monthly rebalance — the hardest possible setting. A negative result here does not falsify the anomaly historically; it says it is not tradeable in this form today.
- **Monthly rebalance is a compromise.** The classic implementation is weekly. The platform's factor pipeline rebalances monthly (`ME`), so the signal is up to four weeks stale by the time positions are refreshed. Expect a weaker result than published weekly numbers.
- **Transaction costs are simplified.** A flat 10 bps of turnover ignores bid-ask spread widening in the very names the strategy buys (stocks that just crashed trade with wide spreads). Real costs are higher, so the net numbers here are optimistic.
- **Short leg is simulated as free.** Borrow fees and recall risk on recently crashed names are ignored; both hurt the short side of this trade in practice.
- **Losers can be losers for a reason.** The factor cannot distinguish a liquidity-driven overshoot from a genuine fundamental collapse (fraud, earnings miss). The long leg systematically holds some of the latter, and the -100% delisting convention makes those maximally painful.
- **Known price-data corruption.** ~87 delisted symbols in `prices.parquet` contain corrupted quotes (e.g. a stock flipping between $0.68 and $11,000 on alternating days), concentrated in 2010–2017. The point-in-time universe filter and the `|rev_21d| < 10` guard remove most but not all of the effect; this is why the backtest window starts in 2018. Windows starting earlier produce meaningless metrics until the panel is cleaned (planned migration to FMP data).
- **Single-window result.** No sub-period or parameter-sensitivity analysis here; a monthly-vs-weekly rebalance comparison would need intraweek rebalance support in `calculate_portfolio_returns`.

## 8. Summary

The verdict comes from the three yes/no questions in Section 6 (run the notebook to see the current numbers against the stored data). As of the initial run: net Sharpe over 2018–2025 was **negative** (≈ -0.2), gross Sharpe was also negative, and the strategy did not beat the momentum reference. The conclusion: the 1-month reversal premium, at monthly rebalance in the modern S&P 500, is **not tradeable** — consistent with Nagel (2012). The strategy stays in the platform as a registered, honestly-documented negative result, which is itself useful as a benchmark for signal-decay studies.